In [1]:
import pandas as pd
import numpy as np

from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)
from scipy.special import softmax

import torch
from torch.utils.data import Dataset

import torch.nn as nn

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

SEED = 42
MAX_LEN = 128

In [2]:
MODELS = [
    "allegro/herbert-base-cased",
    "dkleczek/bert-base-polish-cased-v1",
    "sdadas/polish-roberta-base-v2"
]

In [5]:
df = pd.read_csv("hate_train.csv")

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df["label"]
)

## Ważenie przykładów

In [6]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train_df["label"].values
)

class_weights = torch.tensor(class_weights, dtype=torch.float)
print(class_weights)

tensor([0.5463, 5.8972])


In [7]:


class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")

        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = nn.CrossEntropyLoss(
            weight=class_weights.to(logits.device)
        )

        loss = loss_fct(
            logits.view(-1, model.config.num_labels),
            labels.view(-1)
        )

        return (loss, outputs) if return_outputs else loss

In [8]:

class HateDataset(Dataset):

    def __init__(self, texts, labels, tokenizer):
        self.encodings = tokenizer(
            texts.tolist(),
            truncation=True,
            padding=True,
            max_length=MAX_LEN
        )

        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item["labels"] = torch.tensor(
            self.labels[idx],
            dtype=torch.long
        )

        return item

def compute_metrics(pred):

    labels = pred.label_ids
    preds = pred.predictions.argmax(axis=1)

    acc = accuracy_score(labels, preds)
    precision = precision_score(labels, preds, zero_division=0)
    recall = recall_score(labels, preds, zero_division=0)
    f1 = f1_score(labels, preds, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()

    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    probs = softmax(pred.predictions, axis=1)[:, 1]
    preds = probs >= 0.5

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "f1": f1,
        "roc_auc": roc_auc_score(labels, probs),
        "pr_auc": average_precision_score(labels, probs)
    }

## Porównanie modeli

In [ ]:
results = []

for model_name in MODELS:

    print("=" * 50)
    print(model_name)

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    train_dataset = HateDataset(
        train_df["sentence"],
        train_df["label"].values,
        tokenizer
    )

    val_dataset = HateDataset(
        val_df["sentence"],
        val_df["label"].values,
        tokenizer
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2
    )

    args = TrainingArguments(
        output_dir=f"./tmp_{model_name.split('/')[-1]}",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=5,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_steps=20,
        report_to="none"
    )

    trainer = WeightedTrainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics
    )

    trainer.train()

    metrics = trainer.evaluate()

    results.append({
        "model": model_name,
        "f1": metrics["eval_f1"],
        "accuracy": metrics["eval_accuracy"]
    })

In [ ]:

results_df = pd.DataFrame(results)

print("\nRESULTS")
print(results_df.sort_values("f1", ascending=False))

best_model_name = results_df.sort_values(
    "f1",
    ascending=False
).iloc[0]["model"]

print(f"\nBEST MODEL: {best_model_name}")

## Analiza najlepszego modelu

In [9]:
best_model_name = "dkleczek/bert-base-polish-cased-v1"

In [10]:
print(best_model_name)

tokenizer = AutoTokenizer.from_pretrained(best_model_name)

train_dataset = HateDataset(
        train_df["sentence"],
        train_df["label"].values,
        tokenizer
    )

val_dataset = HateDataset(
    val_df["sentence"],
    val_df["label"].values,
    tokenizer
)

model = AutoModelForSequenceClassification.from_pretrained(
    best_model_name,
    num_labels=2
)

args = TrainingArguments(
    output_dir=f"./best_{best_model_name.split('/')[-1]}",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="recall",
    greater_is_better=True,
    logging_steps=20,
    report_to="none"
)

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

metrics = trainer.evaluate()



dkleczek/bert-base-polish-cased-v1


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/30.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/489k [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,Specificity,F1,Roc Auc,Pr Auc
0.010208,0.899680,5,0.935291,0.638889,0.541176,0.971724,0.585987,0.901846,0.578423


In [11]:
print(metrics)

{'eval_loss': 0.8996803760528564, 'eval_accuracy': 0.9352911896465903, 'eval_precision': 0.6388888888888888, 'eval_recall': 0.5411764705882353, 'eval_specificity': 0.9717237629146275, 'eval_f1': 0.5859872611464968, 'eval_roc_auc': 0.9018456322169977, 'eval_pr_auc': 0.5784225334530249}


In [12]:
from sklearn.metrics import classification_report

predictions = trainer.predict(val_dataset)

y_true = predictions.label_ids
y_pred = predictions.predictions.argmax(axis=1)

print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.96      0.97      0.96      1839
           1       0.64      0.54      0.59       170

    accuracy                           0.94      2009
   macro avg       0.80      0.76      0.78      2009
weighted avg       0.93      0.94      0.93      2009



In [13]:
cm = confusion_matrix(y_true, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"]
)

print(cm_df)

          Predicted 0  Predicted 1
Actual 0         1787           52
Actual 1           78           92


## Najlepszy model

In [ ]:

tokenizer = AutoTokenizer.from_pretrained(best_model_name)

full_dataset = HateDataset(
    df["sentence"],
    df["label"].values,
    tokenizer
)

best_model = AutoModelForSequenceClassification.from_pretrained(
    best_model_name,
    num_labels=2
)

args = TrainingArguments(
    output_dir="./final_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=best_model,
    args=args,
    train_dataset=full_dataset
)

trainer.train()

## Predykcje

In [ ]:
with open("hate_test_data.txt", "r", encoding="utf8") as f:
    test_texts = [line.strip() for line in f]

enc = tokenizer(
    test_texts,
    truncation=True,
    padding=True,
    max_length=MAX_LEN,
    return_tensors="pt"
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

best_model.to(device)

enc = {
    k: v.to(device)
    for k, v in enc.items()
}

with torch.no_grad():
    outputs = best_model(**enc)

preds = outputs.logits.argmax(dim=1).cpu().numpy()

pd.DataFrame(preds).to_csv(
    "pred.csv",
    header=False,
    index=False
)
